# Note on solving equation
Having a problem of form
$ A x = b $
with $ A \in \mathbb{R}^{m \times n} $, $ x \in \mathbb{R}^{n} $, $ b \in \mathbb{R}^{m} $
we can solve it with different methods.
- If $ A $ is square and invertible, we can compute $ x = A^{-1} b $.
- If $ A $ is not square or not invertible, we can solve the least squares problem
  $ \min_x || A x - b ||_2^2 $
  which has solution $ x = (A^T A)^{-1} A^T b $ if $ A^T A $ is invertible.
- If $ A^T A $ is not invertible, we can use the pseudo-inverse
  $ x = A^+ b $
  where $ A^+ = V \Sigma^+ U^T $
  is computed from the SVD of $ A = U \Sigma V^T $.

## Computational complexity
- Computing the inverse of a matrix is $ O(n^3) $.
- Computing the pseudo-inverse with SVD is $ O(\min(mn^2, nm^2)) $.
- Solving the least squares problem with QR decomposition is $ O(mn^2) $.
- Solving the least squares problem with SVD is $ O(\min(mn^2, nm^2)) $.
- Solving the least squares problem with normal equations is $ O(n^3) $.

In [1]:
import numpy as np
import timeit

n_points = 10000
n_dims = 1000

A = np.random.rand(n_points, n_dims)
b = np.random.rand(n_points)


def solve_linalg_solve(A,b):
    """
    Solve the normal equations using the generic solver.
    Equivalent to: x = (AᵀA)⁻¹ Aᵀb
    """
    return np.linalg.solve(A.T @ A, A.T @ b)

def solve_lstsq(A, b):
    """
    Use NumPy's least-squares routine. Internally calls LAPACK
    DGELSS (QR decomposition). 
    """
    return np.linalg.lstsq(A, b, rcond=None)[0]

def solve_pinv(A, b):
    """
    Compute the pseudoinverse explicitly: x = A⁺ b
    """
    return np.linalg.pinv(A) @ b


n_runs = 10

t_solve   = timeit.timeit('solve_linalg_solve(A,b)', globals=globals(), number=n_runs)
t_lstsq   = timeit.timeit('solve_lstsq(A,b)', globals=globals(), number=n_runs)
t_pinv    = timeit.timeit('solve_pinv(A,b)', globals=globals(), number=n_runs)

print(f"np.linalg.solve   (normal equations)    : {t_solve:.4f}s")
print(f"np.linalg.lstsq   (QR/SVD)              : {t_lstsq:.4f}s")
print(f"Pseudoinverse (pinv @ b)                : {t_pinv:.4f}s")

np.linalg.solve   (normal equations)    : 1.3597s
np.linalg.lstsq   (QR/SVD)              : 11.4263s
Pseudoinverse (pinv @ b)                : 24.6129s


In [2]:
x_solve   = solve_linalg_solve(A,b)
x_lstsq   = solve_lstsq(A,b)

# The solutions should be very close
print("‖x_solve - x_lstsq‖₂ =", np.linalg.norm(x_solve - x_lstsq))

‖x_solve - x_lstsq‖₂ = 3.05947153299184e-13
